In [ ]:
"""
Reproducibility artifact for the paper's examples and figures.
 Paper: A General Framework for Consensus in Opinion Dynamics under
    Changing Influence, Bias, and Partial Synchrony
     by Juan PAZ, Sophia KNIGHT,  Camilo ROCHA, Frank VALENCIA.

This script reproduces all numerical simulations reported in the paper.
The examples referenced below are identified by their titles in the
paper:

  * "A running example" (Section: The Model), parts (1) through (4):
      (1) Classical DeGroot run.
      (2) Gossip run over the omega-word ((2,1)(2,3)(1,2)(3,2))^omega.
      (3) Receptive-resistant-bias (confirmation-bias) run.
      (4) Partially synchronous run alternating extreme and middle updates.
      All four share the same 3-agent AGOTS with x0 = (0.2, 0.4, 0.9).
      The script prints the initial steps of each run and its limit
      consensus value.

  * "Dissensus Remark" (Remark in Section: The Model):
      Gossip run over the strongly-fair word
          w = prod_{n=1}^{infty} ( (1,2)^n (2,1) (3,2)^n (2,3) ).
      Produces the non-consensus figure (file: nonconsensus.pdf).

  * "Bounded communication" example (Section: Consensus Results):
      Part (1) coincides with the partially-synchronous run above;
      part (2) coincides with the fading-influence gossip run above.
      No additional simulation is required.

  * "Consensus from Generating Set"
    (Example in Section: Consensus Results):
      6-agent AGOTS with two disjoint source components, each containing
      a puppet agent without self-influence. Partially synchronous
      scheduler combined with confirmation-biased weights. Produces
      the opinion-evolution figure (file: genset_puppet.pdf).

Usage
-----
From the command line, each simulation has its own subcommand:

    python paper_examples.py running       # the four variants of the running example
    python paper_examples.py nonconsensus  # the dissensus remark and its figure
    python paper_examples.py genset        # the consensus-from-generating-set example and its figure
    python paper_examples.py                # (no argument) runs all three, same as 'all'
    python paper_examples.py all           # explicit 'all'

From a Jupyter or Google Colab notebook, upload the file and import it:

    import paper_examples
    paper_examples.run_running_example()      # or run_nonconsensus() / run_genset()
    paper_examples.main()                      # runs all three

    # or, to select a specific example programmatically:
    paper_examples.main(['genset'])

Requirements
------------
    python >= 3.8
    numpy
    matplotlib

No other dependencies. The figures are saved as PDF in the current
working directory.

Conventions
-----------
Agents are indexed from 1 in the paper and from 0 in the code. Each
influence graph G = (A, E, I) is represented by a numpy matrix
I_base of shape (N, N), where I_base[b, a] = I(b, a) is the weight
with which agent b influences agent a (so columns sum to 1). The
influence matrix at time t, once schedulers and biases are applied,
is built row by row so that each A^t @ x gives the next opinion
vector.
"""

from __future__ import annotations

import argparse
import sys

import numpy as np
import matplotlib.pyplot as plt


# =============================================================================
# Utilities
# =============================================================================

def bias_ratio(z, delta):
    """
    Ratio beta(z)/z for the confirmation-bias function
        beta(z) = z * (1 + delta - |z|) / (1 + delta).
    Returns the well-defined limit (1) at z = 0, but callers should handle
    z == 0 separately to match the bias scheduler's convention of setting
    effective weight to 0 in that case.
    """
    return (1.0 + delta - abs(z)) / (1.0 + delta)


def build_row_stochastic(I_base, active_edges, x, deltas=None):
    """
    Build the row-stochastic transition matrix A^t applied to x.

    Parameters
    ----------
    I_base : (N, N) array
        Base influence graph, with I_base[b, a] = I(b, a).
    active_edges : list of (b, a)
        The non-self edges activated at this time step. Agents whose
        incoming-edge row is empty do not update.
    x : (N,) array
        Current opinion vector.
    deltas : dict or None
        If given, bias parameters keyed by (b, a); effective weight on
        (b, a) is I_base[b, a] * bias_ratio(x[b] - x[a], deltas[(b, a)]).
        If None, no bias is applied and the effective weight is I_base[b, a].

    Returns
    -------
    A : (N, N) row-stochastic array
        The transition matrix A^t for this step, consistent with the
        AGOTS update rule x' = A x in the code's convention
        (row-stochastic; A[a, b] is the weight with which agent b
        influences agent a at this step).
    """
    N = I_base.shape[0]
    A = np.eye(N)
    for a in range(N):
        incoming = [(b, aa) for (b, aa) in active_edges if aa == a]
        if not incoming:
            continue  # agent a does not update this step
        row = np.zeros(N)
        for (b, _) in incoming:
            if deltas is None:
                eff = I_base[b, a]
            else:
                z = x[b] - x[a]
                if abs(z) < 1e-15:
                    eff = 0.0
                else:
                    eff = I_base[b, a] * bias_ratio(z, deltas[(b, a)])
            row[b] = eff
        row[a] = 1.0 - row.sum()
        A[a, :] = row
    assert np.allclose(A.sum(axis=1), 1.0), (
        "Internal error: constructed matrix is not row-stochastic"
    )
    return A


# =============================================================================
# "A running example": 3-agent AGOTS, four scheduler variants
# =============================================================================

def running_example_graph():
    """
    Influence graph G of the running example.
    Agents A = {1, 2, 3} (indices 0, 1, 2).
    I[1,1] = I[2,1] = 0.5
    I[1,2] = I[3,2] = 0.25,  I[2,2] = 0.5
    I[2,3] = I[3,3] = 0.5
    """
    N = 3
    I = np.zeros((N, N))
    I[0, 0] = 0.5
    I[1, 0] = 0.5
    I[1, 1] = 0.5
    I[0, 1] = 0.25
    I[2, 1] = 0.25
    I[2, 2] = 0.5
    I[1, 2] = 0.5
    assert np.allclose(I.sum(axis=0), 1.0)
    return I


def run_degroot(T=50):
    """
    Part (1) of the running example: classical DeGroot run. Every non-self
    edge of G is active at every step. The update is x' = A x with
    A[a, b] = I[b, a].
    """
    I = running_example_graph()
    x0 = np.array([0.2, 0.4, 0.9])
    A = I.T.copy()
    assert np.allclose(A.sum(axis=1), 1.0)
    traj = [x0.copy()]
    x = x0.copy()
    for _ in range(T):
        x = A @ x
        traj.append(x.copy())
    return np.array(traj)


def run_gossip(omega_word, T):
    """
    Gossip run. At step t, the scheduler activates exactly one non-self edge
    omega_word[t % len(omega_word)] = (b, a) with zero-indexed agents.
    Agent a updates by x_a' = (1 - I(b,a)) x_a + I(b,a) x_b; all others
    keep their opinion.
    """
    I = running_example_graph()
    x0 = np.array([0.2, 0.4, 0.9])
    traj = [x0.copy()]
    x = x0.copy()
    for t in range(T):
        b, a = omega_word[t % len(omega_word)]
        w = I[b, a]
        x_new = x.copy()
        x_new[a] = (1 - w) * x[a] + w * x[b]
        x = x_new
        traj.append(x.copy())
    return np.array(traj)


def run_bias(T=50):
    """
    Part (3) of the running example: receptive-resistant-bias
    (confirmation-bias) run over G. Bias parameters from the paper:
      delta_{1,2} = 0.01   (the function beta_{1,2}, on edge (2,1))
      delta_{2,1} = 0.015  (beta_{2,1}, on edge (1,2))
      delta_{2,3} = 0.02   (beta_{2,3}, on edge (3,2))
      delta_{3,2} = 0.025  (beta_{3,2}, on edge (2,3))
    """
    I = running_example_graph()
    x0 = np.array([0.2, 0.4, 0.9])
    deltas = {
        (1, 0): 0.01,
        (0, 1): 0.015,
        (2, 1): 0.02,
        (1, 2): 0.025,
    }
    active_edges = list(deltas.keys())
    traj = [x0.copy()]
    x = x0.copy()
    for _ in range(T):
        A = build_row_stochastic(I, active_edges, x, deltas=deltas)
        x = A @ x
        traj.append(x.copy())
    return np.array(traj)


def run_partial_sync(T=50):
    """
    Part (4) of the running example: partially synchronous run. Even steps
    activate the incoming edges of agents 1 and 3; odd steps activate the
    incoming edges of agent 2.
      even t: edges {(2,1), (2,3)}
      odd t:  edges {(1,2), (3,2)}
    """
    I = running_example_graph()
    x0 = np.array([0.2, 0.4, 0.9])
    edges_even = [(1, 0), (1, 2)]
    edges_odd = [(0, 1), (2, 1)]
    traj = [x0.copy()]
    x = x0.copy()
    for t in range(T):
        active = edges_even if t % 2 == 0 else edges_odd
        A = build_row_stochastic(I, active, x, deltas=None)
        x = A @ x
        traj.append(x.copy())
    return np.array(traj)


def run_running_example():
    """
    Reproduce the numerical steps of the running example and its four
    schedulers, and report the consensus limits cited in the paper.
    """
    print("=" * 70)
    print('Example: "A running example", parts (1) through (4)')
    print("=" * 70)

    print("\n(1) Classical DeGroot run, first four steps:")
    traj = run_degroot(T=500)
    for t in range(5):
        print(f"  t={t}: {traj[t].round(4)}")
    print(f"  t=500: {traj[-1].round(6)}   (paper reports 0.475)")

    print("\n(2) Gossip run over omega-word ((2,1)(2,3)(1,2)(3,2))^omega,")
    print("    first four steps:")
    omega_word = [(1, 0), (1, 2), (0, 1), (2, 1)]
    traj = run_gossip(omega_word, T=100000)
    for t in range(5):
        print(f"  t={t}: {traj[t].round(4)}")
    print(f"  t=100000: {traj[-1].round(6)}")
    print(f"  Paper reports 53/115 = {53/115:.6f}")

    print("\n(3) Receptive-resistant-bias (confirmation-bias) run,")
    print("    first four steps:")
    traj = run_bias(T=500)
    for t in range(5):
        print(f"  t={t}: {traj[t].round(4)}")
    print(f"  t=500: {traj[-1].round(6)}   (paper reports approx. 0.475)")

    print("\n(4) Partially synchronous run, first four steps:")
    traj = run_partial_sync(T=500)
    for t in range(5):
        print(f"  t={t}: {traj[t].round(4)}")
    print(f"  t=500: {traj[-1].round(6)}   (paper reports 0.45)")


# =============================================================================
# Dissensus Remark: fading-influence gossip run
# =============================================================================

def nonconsensus_word(blocks):
    """
    The omega-word w = prod_{n=1}^{blocks} ( (1,2)^n (2,1) (3,2)^n (2,3) )
    truncated to a finite number of blocks. Returned as a list of
    (b, a) edges in 0-indexed form.
    """
    edge_1_2 = (0, 1)
    edge_2_1 = (1, 0)
    edge_3_2 = (2, 1)
    edge_2_3 = (1, 2)

    word = []
    for n in range(1, blocks + 1):
        word.extend([edge_1_2] * n)
        word.append(edge_2_1)
        word.extend([edge_3_2] * n)
        word.append(edge_2_3)
    return word


def run_nonconsensus(save_path="nonconsensus.pdf"):
    """
    Reproduce the gossip run of the Dissensus Remark under the
    strongly-fair word
        w = prod_{n=1}^{infty} ( (1,2)^n (2,1) (3,2)^n (2,3) )
    truncated to enough blocks to cover t = 500. Save the resulting
    plot as 'nonconsensus.pdf'.
    """
    print("=" * 70)
    print('Dissensus Remark')
    print("=" * 70)

    blocks = 30
    word = nonconsensus_word(blocks)
    T = 500
    traj = run_gossip(word, T)

    print(f"\nFinal at t={T}: {traj[-1].round(6)}")
    print(f"Pairwise gaps at t={T}:")
    print(f"  |x1 - x2| = {abs(traj[-1, 0] - traj[-1, 1]):.6f}")
    print(f"  |x2 - x3| = {abs(traj[-1, 1] - traj[-1, 2]):.6f}")
    print(f"  |x1 - x3| = {abs(traj[-1, 0] - traj[-1, 2]):.6f}")
    print("Paper reports: agents 2 and 3 near 0.503, agent 1 near 0.444,")
    print("gap approximately 0.059.")

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    colors = ["#d62728", "#2ca02c", "#1f77b4"]
    labels = ["agent 1", "agent 2", "agent 3"]
    for i in range(3):
        ax.plot(range(T + 1), traj[:, i], color=colors[i],
                label=labels[i], linewidth=1.6)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"opinion $\mathbf{x}_a^t$")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc="center right")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    print(f"\nSaved {save_path}")


# =============================================================================
# Consensus from Generating Set: 6-agent partial-sync + bias example
# =============================================================================

def genset_graph():
    """
    Influence graph for the Consensus from Generating Set example: 6 agents,
    two disjoint source components {1,2} and {3,4}, both containing a
    puppet agent (no self-influence), feeding two receptives that also
    interact with each other.

    Weights:
        I[1,1] = I[2,1] = 0.5
        I[1,2] = 1,  I[2,2] = 0                (agent 2 is puppet of 1)
        I[3,3] = I[4,3] = 0.5
        I[3,4] = 1,  I[4,4] = 0                (agent 4 is puppet of 3)
        I[5,5] = 0.3, I[1,5] = I[3,5] = 0.25, I[6,5] = 0.2
        I[6,6] = 0.3, I[2,6] = I[4,6] = 0.25, I[5,6] = 0.2
    """
    N = 6
    I = np.zeros((N, N))
    I[0, 0] = 0.5; I[1, 0] = 0.5
    I[1, 1] = 0.0; I[0, 1] = 1.0
    I[2, 2] = 0.5; I[3, 2] = 0.5
    I[3, 3] = 0.0; I[2, 3] = 1.0
    I[4, 4] = 0.3; I[0, 4] = 0.25; I[2, 4] = 0.25; I[5, 4] = 0.2
    I[5, 5] = 0.3; I[1, 5] = 0.25; I[3, 5] = 0.25; I[4, 5] = 0.2
    assert np.allclose(I.sum(axis=0), 1.0)
    return I


def run_genset(save_path="genset_puppet.pdf"):
    """
    Reproduce the example titled "Consensus from Generating Set":
    partially synchronous + confirmation-bias scheduler on a 6-agent
    graph with two puppet source components.

    Scheduler:
      even t: activate source-internal edges {(1,2),(2,1),(3,4),(4,3)}
      odd  t: activate edges into and between receptives
              {(1,5),(3,5),(6,5),(2,6),(4,6),(5,6)}

    Confirmation bias with delta = 0.05 on every non-self edge.
    """
    print("=" * 70)
    print('Example: "Consensus from Generating Set"')
    print("=" * 70)

    I = genset_graph()
    x0 = np.array([0.2, 0.8, 0.5, 0.2, 0.5, 0.9])

    edges_even = [(0, 1), (1, 0), (2, 3), (3, 2)]
    edges_odd = [
        (0, 4), (2, 4), (5, 4),
        (1, 5), (3, 5), (4, 5),
    ]

    delta_value = 0.05
    all_non_self = edges_even + edges_odd
    deltas = {edge: delta_value for edge in all_non_self}

    T = 30
    traj = [x0.copy()]
    x = x0.copy()
    for t in range(T):
        active = edges_even if t % 2 == 0 else edges_odd
        A = build_row_stochastic(I, active, x, deltas=deltas)
        x = A @ x
        traj.append(x.copy())
    traj = np.array(traj)

    limit_12 = (2 * x0[0] + x0[1]) / 3
    limit_34 = (2 * x0[2] + x0[3]) / 3
    print(f"\nPredicted limit of source {{1, 2}}: {limit_12}")
    print(f"Predicted limit of source {{3, 4}}: {limit_34}")
    print(f"Averages match (by construction): {np.isclose(limit_12, limit_34)}")

    print(f"\nFinal at t={T}: {traj[-1].round(6)}")
    print(f"Max pairwise gap at t={T}: {traj[-1].max() - traj[-1].min():.2e}")

    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    colors = ["#d62728", "#ff7f0e", "#2ca02c", "#17becf", "#1f77b4", "#9467bd"]
    labels = ["agent 1 (F)", "agent 2", "agent 3 (F)", "agent 4",
              "agent 5", "agent 6"]
    styles = ["-", "-", "-", "-", "--", "--"]
    widths = [1.8, 1.2, 1.8, 1.2, 1.4, 1.4]
    for i in range(6):
        ax.plot(range(T + 1), traj[:, i], styles[i], color=colors[i],
                label=labels[i], linewidth=widths[i])
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"opinion $\mathbf{x}_a^t$")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc="center right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    print(f"\nSaved {save_path}")


# =============================================================================
# Command-line interface
# =============================================================================

def main(argv=None):
    """
    Entry point. When called from the command line, parses sys.argv.
    When called from a notebook (Jupyter, Colab, IPython) the kernel's
    connection-file arguments are ignored and the default 'all' is used.
    You can also call main(['running']), main(['genset']), etc.
    explicitly from a notebook cell.
    """
    parser = argparse.ArgumentParser(
        description="Reproduce the paper's numerical examples and figures."
    )
    parser.add_argument(
        "command",
        nargs="?",
        default="all",
        choices=["running", "nonconsensus", "genset", "all"],
        help=(
            "Which simulation to run. "
            "'running' reproduces the four parts of the Running Example. "
            "'nonconsensus' reproduces the Dissensus Remark and saves "
            "nonconsensus.pdf. "
            "'genset' reproduces the Consensus from Generating Set "
            "example and saves genset_puppet.pdf. "
            "'all' runs everything (default)."
        ),
    )

    if argv is None:
        in_notebook = (
            "ipykernel" in sys.modules
            or "google.colab" in sys.modules
        )
        argv = [] if in_notebook else sys.argv[1:]

    args = parser.parse_args(argv)

    if args.command in ("running", "all"):
        run_running_example()
        print()
    if args.command in ("nonconsensus", "all"):
        run_nonconsensus()
        print()
    if args.command in ("genset", "all"):
        run_genset()
        print()


if __name__ == "__main__":
    main()